# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

In [ ]:
git add .
git commit -m "Add INFO4670 assignment files"
git push -u origin main

## Setup — load the three files (given)

In [3]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [4]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()
blank_cols = students.columns[students.isna().any()].tolist()


**Your justification (1–2 sentences):** I would use the median imputation method for `study_hours_reported` because the column is numeric and right-skewed with a moderate number of missing values (255 missing out of 2027 rows). Median is robust to skewness and outliers, so it preserves the central tendency without distorting the distribution as much as a mean would.

In [5]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [6]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
students = students.copy()
students["housing_clean"] = (
    students["housing"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", " ")
    .str.replace("_", " ")
    .map({
        "off campus": "Off-Campus",
        "on campus": "On-Campus",
        "with family": "With Family",
    })
)

# Confirm the standardized groups are exactly 3
print(students["housing_clean"].value_counts(dropna=False).to_dict())


{'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


In [7]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [8]:
# TODO
impossible_ages = sorted(students.loc[~students['age'].between(0, 120), 'age'].dropna().unique().tolist())
n_neg_work = int((students['work_hours_per_week'] < 0).sum())
print('Impossible ages:', impossible_ages)
print('Negative work rows:', n_neg_work)


Impossible ages: [-22, 199, 220]
Negative work rows: 4


In [9]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [10]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset='student_id', keep='first').copy()
print('rows after dedup:', len(students_dedup))
print('unique student_id:', students_dedup['student_id'].is_unique)


rows after dedup: 2000
unique student_id: True


**Why not de-dupe enroll / activity? (1 sentence):** Each row in `course_enrollments` represents one enrollment event and each row in `weekly_activity` represents one weekly activity record, so removing duplicates by student ID would erase legitimate repeated events rather than repeated student identities.

In [11]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [12]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
students_dedup = students_dedup.copy()
students_dedup['student_id_num'] = students_dedup['student_id'].str.replace('NU-', '', regex=False).astype(int)

activity_summary = activity.groupby('student_id', as_index=False).agg(
    total_minutes_active=('minutes_active', 'sum'),
    total_logins=('week', 'count')
)

analysis = students_dedup.merge(
    activity_summary,
    left_on='student_id',
    right_on='student_id',
    how='left'
)
analysis = analysis.rename(columns={'student_id_num': 'student_id_num'})
analysis['student_id'] = analysis['student_id'].astype(str)
analysis['student_id_num'] = analysis['student_id'].str.replace('NU-', '', regex=False).astype(int)
print('analysis rows', len(analysis), 'unique', analysis['student_id'].is_unique)
print(analysis.head().to_string())


analysis rows 2000 unique True
  student_id                   major  age      housing  commute_miles  work_hours_per_week  advisor_meetings  credits_attempted  final_gpa  study_hours_reported enrollment_date  dropped_out housing_clean  student_id_num  total_minutes_active  total_logins
0  NU-101508            Data Science   27   Off-Campus            7.6                   15                 1                 81       1.86                   NaN      2023-09-13            1    Off-Campus          101508                  2750            16
1  NU-102438   Learning Technologies   25  With Family           14.3                   35                 1                100       2.06                   9.5      2023-10-19            0   With Family          102438                  2792            16
2  NU-102385   Learning Technologies   26  With Family            3.4                   18                 9                 77       3.41                  19.1      11/16/2022            0   With Fami

In [13]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [14]:
# TODO
matched = int(enroll['sid'].isin(set(analysis['student_id_num'])).sum())
print('matched rows:', matched)
print('orphan ids:', len(set(enroll['sid']) - set(analysis['student_id_num'])))


matched rows: 8041
orphan ids: 14


In [15]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [16]:
# TODO
dates_parsed = pd.to_datetime(students_dedup['enrollment_date'], format='mixed', errors='coerce')
print('parsed NaT count:', dates_parsed.isna().sum())
print('before blanks', students_dedup['enrollment_date'].isna().sum())
print('after blanks', dates_parsed.isna().sum())


parsed NaT count: 0
before blanks 0
after blanks 0


In [17]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [18]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
analysis = analysis.copy()
analysis['study_z'] = (analysis['study_hours_reported'] - analysis['study_hours_reported'].mean()) / analysis['study_hours_reported'].std(ddof=1)
analysis['gpa_band'] = pd.cut(
    analysis['final_gpa'],
    bins=[-0.01, 1, 2, 3, 4],
    labels=['0-1', '1-2', '2-3', '3-4'],
    include_lowest=True
)
print(analysis[['study_z', 'gpa_band']].head().to_string())
print('gpa_band unique:', analysis['gpa_band'].nunique())
print('study_z mean:', analysis['study_z'].mean())


    study_z gpa_band
0       NaN      1-2
1 -0.251817      2-3
2  2.241895      3-4
3 -0.615483      1-2
4 -1.005126      2-3
gpa_band unique: 4
study_z mean: 4.3442403804347476e-16


In [19]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [20]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv('northgate_clean.csv', index=False)
print('wrote northgate_clean.csv')


wrote northgate_clean.csv


In [21]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing `study_hours_reported` → I counted the blanks and would use median imputation because the column is numeric and skewed, so the median is less distorted by a few high values than the mean.
- Housing standardization → I normalized all variants to `Off-Campus`, `On-Campus`, and `With Family` so the categories matched the true groups and had no case/spacing inconsistencies.
- Impossible values → I flagged ages outside 0–120 and negative work hours as invalid because they cannot represent real student records; extreme but valid values (for example, a long commute or a high but possible work week) are kept.
- Duplicate student records → I dropped exact duplicate rows by `student_id` because the student table should be one row per student, while duplicates in other tables are legitimate repeated events.
- Key standardization → I removed the `NU-` prefix from `student_id` before joining to course enrollments, which made the numeric student keys consistent across tables.
- Date parsing → I coerced mixed date strings into pandas datetimes using `format="mixed"` so valid dates across formats were preserved without silently dropping them.


### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [22]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
mean_gpa = analysis['final_gpa'].mean()
print('Mean GPA:', mean_gpa)
print('Correlation final_gpa vs study_hours_reported:', analysis['final_gpa'].corr(analysis['study_hours_reported']))
print('Average GPA by housing_clean:\n', analysis.groupby('housing_clean')['final_gpa'].mean().to_string())


Mean GPA: 2.3053600000000003
Correlation final_gpa vs study_hours_reported: 0.693921709709522
Average GPA by housing_clean:
 housing_clean
Off-Campus     2.329925
On-Campus      2.276164
With Family    2.293285


In [23]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
